# 03 - Regressao Logistica 
Modelo **baseline**. Protocolo de avaliacao do projeto:
1. treina em `treino` -> avalia em `teste1`;
2. junta `teste1` ao treino -> avalia em `teste2`.

Para o desbalanceamento usamos `class_weight='balanced'` 

In [ ]:
# bibliotecas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.options.display.float_format = '{:.3f}'.format
import sys; sys.path.append("../src")   # acesso a preprocessing.py
DATA = "../data/processed"


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num = ["idade_anos", "NU_CONTATO", "dias_diag_trat"]
cat = ["CS_SEXO","CS_RACA","CS_ESCOL_N","SG_UF_NOT","RAIOX_TORA","TESTE_TUBE",
       "BACILOSC_E","HIV","AGRAVAIDS","AGRAVALCOO","AGRAVDIABE","AGRAVDOENC",
       "AGRAVDROGA","AGRAVTABAC","TRAT_SUPER","ANT_RETRO","BENEF_GOV",
       "POP_RUA","POP_LIBER","POP_IMIG","POP_SAUDE"]

def preparar(df):
    # cria o atraso ate o inicio do tratamento e deixa as categoricas como texto
    df = df.copy()
    df["DT_DIAG"]    = pd.to_datetime(df["DT_DIAG"], errors="coerce")
    df["DT_INIC_TR"] = pd.to_datetime(df["DT_INIC_TR"], errors="coerce")
    df["dias_diag_trat"] = (df["DT_INIC_TR"] - df["DT_DIAG"]).dt.days
    for c in cat:
        df[c] = df[c].fillna("ignorado").astype(str)
    return df[num + cat]
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, RocCurveDisplay

treino = pd.read_csv(f"{DATA}/treino.csv"); teste1 = pd.read_csv(f"{DATA}/teste1.csv"); teste2 = pd.read_csv(f"{DATA}/teste2.csv")
print("treino:", len(treino), "linhas (base completa)")

## Metricas escolhidas 
Como o alvo e desbalanceado, **acuracia engana**. Priorizamos **recall** (achar quem
vai abandonar), **F1** e **ROC-AUC**.

In [ ]:
# joga as metricas numa tabelinha, fica mais facil de ler
def tabela_metricas(y_true, y_pred, y_proba, nome="Modelo"):
    return pd.DataFrame({
        "acuracia":[accuracy_score(y_true, y_pred)],
        "recall":[recall_score(y_true, y_pred)],
        "precisao":[precision_score(y_true, y_pred)],
        "f1":[f1_score(y_true, y_pred)],
        "roc_auc":[roc_auc_score(y_true, y_proba)],
    }, index=[nome])

### Treina no treino, avalia no teste1

In [ ]:
# pre-processa os dados, treina com peso de classe (pra lidar com o desbalanceamento)
transf = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
X_train = transf.fit_transform(preparar(treino)); y_train = treino["ltfu"]

logreg = LogisticRegression(max_iter=300, class_weight="balanced")
logreg.fit(X_train, y_train)

# agora mede no teste1
X_teste1 = transf.transform(preparar(teste1)); y_teste1 = teste1["ltfu"]
proba1 = logreg.predict_proba(X_teste1)[:,1]; pred1 = (proba1 >= 0.5).astype(int)
m1 = tabela_metricas(y_teste1, pred1, proba1, "LogReg - Teste1"); m1

In [ ]:
RocCurveDisplay.from_predictions(y_teste1, proba1, plot_chance_level=True)
plt.title("Curva ROC - Regressao Logistica (Teste1)"); plt.show()

### Junta teste1 ao treino e avalia no teste2

In [ ]:
# junta o teste1 ao treino e roda de novo
treino2 = pd.concat([treino, teste1], ignore_index=True)
transf2 = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False), cat),
], verbose_feature_names_out=False)
X_tr2 = transf2.fit_transform(preparar(treino2)); y_tr2 = treino2["ltfu"]
logreg2 = LogisticRegression(max_iter=300, class_weight="balanced").fit(X_tr2, y_tr2)

X_teste2 = transf2.transform(preparar(teste2)); y_teste2 = teste2["ltfu"]
proba2 = logreg2.predict_proba(X_teste2)[:,1]; pred2 = (proba2 >= 0.5).astype(int)
m2 = tabela_metricas(y_teste2, pred2, proba2, "LogReg - Teste2"); m2

## Interpretacao dos parametros 

In [ ]:
# coeficiente positivo puxa pro abandono; negativo protege
nomes = transf2.get_feature_names_out()
coef = pd.Series(logreg2.coef_[0], index=nomes).sort_values()
pd.concat([coef.head(8), coef.tail(8)]).plot(kind="barh", figsize=(7,6), color="#4c72b0")
plt.title("Coeficientes da regressao logistica"); plt.tight_layout(); plt.show()

In [ ]:
# salva o modelo pra reaproveitar depois
import joblib, os
os.makedirs("../models", exist_ok=True)
joblib.dump(logreg2, "../models/modelo_logreg.joblib")
joblib.dump(transf2, "../models/transformadores.joblib")
print("modelo salvo.")